#5 HERENCIA MULTIPLE

Mejoremos la jerarquía de personas del código anterior. Se ha incluido una función `ingresos` que devuelve el total de ingresos (por defecto devolverá 0)

In [7]:
from datetime import datetime, date

class Persona:
    def __init__(self, nombre: str, dia: int, mes: int, año: int):
        self.nombre = nombre
        self.fecha_nacimiento = date(año, mes, dia)
    def edad(self) -> int:
        hoy = datetime.now().date()
        edad = hoy.year - self.fecha_nacimiento.year
        if (hoy.month, hoy.day) < (self.fecha_nacimiento.month, self.fecha_nacimiento.day):
            #estamos en una fecha anterior a la de mi cumpleaños
            edad -= 1
        return edad
    def ingresos(self) -> float:
        return 0.0
    def __str__(self):
        return f"Soy {type(self).__name__} mi nombre es {self.nombre} tengo {self.edad()} años. Mis ingresos son {self.ingresos()}"

De esta clase Persona heredan entonces tres clases Estudiante (que podría tener como ingresos un estipendio), Trabajador (que tiene como ingresos un salario) y Jubilado (que tiene como ingresos una pensión)

In [10]:
class Estudiante(Persona):
    def __init__(self, nombre: str, dia: int, mes: int, año: int, e: float = 200.0):
        super().__init__(nombre, dia, mes, año)
        self.estipendio = e
    def ingresos(self) -> float:
        return self.estipendio

class Trabajador(Persona):
    def __init__(self, nombre: str, dia: int, mes: int, año: int, s: float):
        super().__init__(nombre, dia, mes, año)
        self.salario = s
        print('ejecutando init de Trabajador')
    def ingresos(self) -> float:
        return self.salario

class Jubilado(Persona):
    def __init__(self, nombre: str, dia: int, mes: int, año: int, p: float):
        super().__init__(nombre, dia, mes, año)
        self.pension = p
        print('ejecutando init de Jubilado')
    def ingresos(self) -> float:
        return self.pension

t = Trabajador("Juan", 18, 3, 1979,10000)
e = Estudiante("Rocío", 31, 1, 2009)
j = Jubilado("Luis", 31, 12, 1960, 4000)
print(t)
print(j)
print(e)

ejecutando init de Trabajador
ejecutando init de Jubilado
Soy Trabajador mi nombre es Juan tengo 47 años. Mis ingresos son 10000
Soy Jubilado mi nombre es Luis tengo 65 años. Mis ingresos son 4000
Soy Estudiante mi nombre es Rocío tengo 17 años. Mis ingresos son 200.0


## 5.1 Heredando multiple
¿Qué hacer si queremos definir un tipo que combine características de varios tipos?

Vamos a definir una clase `Jubilado_Trabajador` para representar a una persona que además de tener una jubilación tiene también un trabajo asalariado. En este caso vamos a heredar de dos clases, de `Jubilado` y de `Trabajador` y por tanto debe tener las dos variables una para salario y otra para pension

In [29]:
#OPCION 1 Llamando directo al __init__ de Persona
class Jubilado_Trabajador(Jubilado, Trabajador):
    def __init__(self, nombre, dia, mes, año, p, s):
        Persona.__init__(self, nombre, dia, mes, año) #Llama directamente al __init__ de Persona no al del super
        self.pension = p
        self.salario = s
    def ingresos(self) -> float:
        return Jubilado.ingresos(self) + Trabajador.ingresos(self)
#
# OPCION 2 No funciona. Llamando directo al __init__ de Jubilado y de Trabajador que son quienes llamaráin al __init__ se super (Persona)
# class Jubilado_Trabajador(Jubilado, Trabajador):
#     def __init__(self, nombre, dia, mes, año, p, s):
#         Jubilado.__init__(self, nombre, dia, mes, año, p) #Llama directamente al __init__ de Jubilado pasando los parámetros de Persona y el parámetro p que debe ser la pensión
#         Trabajador.__init__(self, nombre, dia, mes, año, s) #Llama directamente al __init__ de Trabajador pasando los parámetros de Persona y el parámetro s que debe ser el salario
#     def ingresos(self) -> float:
#         return Jubilado.ingresos(self) + Trabajador.ingresos(self)

#OPCION 3 No funciona. Llamando directo al __init__ de Jubilado y de Trabajador que son quienes llamaráin al __init__ se super (Persona)
# class Jubilado_Trabajador(Jubilado, Trabajador):
#     def __init__(self, nombre, dia, mes, año, p, s):
#         super().__init__(self, nombre, dia, mes, año, p, s)
#     def ingresos(self) -> float:
#         return Jubilado.ingresos(self) + Trabajador.ingresos(self)

# solo_jubilado = Jubilado("Luis", 31, 12, 1960, 4000)
# print(solo_jubilado)
# solo_profesor = Trabajador("Miguel", 31, 12, 1970, 12000)
# print(solo_profesor)
jubilado_y_profesor = Jubilado_Trabajador("Juan", 18, 3, 1979,4000, 12000)
print(jubilado_y_profesor)


__init__ de Persona
Soy Jubilado_Trabajador mi nombre es Juan tengo 47 años. Mis ingresos son 16000


En la opción 1 de la clase anterior al inicializar una instancia de `Jubilado_Trabajador` con `__init__(self, nombre, dia, mes, año, p, s)` se comienza llamando directamente a `Persona.__init__(self, nombre, dia, mes, año)` para que la instancia se inicialice como `Persona` y después se actualizan las variables `pension` y `salario` de modo que la nueva instancia queda inicializada con todas sus variables: las que tiene como `Persona`, las que tiene como `Jubilado` y las que tiene como `Trabajador`.
 Las opciones 2 y 3 dan error por la forma en que en la herencia múltiple Python hace las llamadas a los métodos en las clases ancestros cuando hay métodos con el mismo nombre en la jerarquía

La siguiente forma de definir la jerarquía funciona correctamente. Veamos por qué.

Cuando se crea una instancia de una clase formada por la herencia múltiple `Jubilado_Trabajador(Jubilado, Trabajador)` se invoca al `__init__` siguiente

`def __init__(self, nombre, dia, mes, año, p, s):
        super().__init__(
            nombre,
            dia,
            mes,
            año,
            pension=p,
            salario=s
        )`

Aquí se llama a `super().__init__` pero la clase hereda de `Jubilado` y de `Trabajador`, a cuál de los **super** se estaría llamando aquí?. Cuando Python llama a un método` f `de una instancia de una clase `B` que hereda múltiple de varias clases `A1, A2, ... An` llama realmente al primer método` f `que aparezca definido en una de sus clases padre (ancestros) siguiendo el orden `A1, A2 ...`. y luego a las restantes definiciones de ` m`que puedan estar en las restantes clases `A2, ...An`. En este ejemplo, como la clase hereda de `Jubilado` y `Trabajador` el
        `super().__init__(
            nombre,
            dia,
            mes,
            año,
            pension=p,
            salario=s
        )`

llamará al `__init__ `definido en `Jubilado`

el `__init__ de Jubilado` es

`class Jubilado(Persona):
    def __init__(self, *args, pension, **kwargs):
        print("__init__ de Jubilado")
        print(f'pension: {pension} **kargs = {kwargs}')
        self.pension = pension
        super().__init__(*args, **kwargs) #paso como diccionario los mismos parámetros nombrados que me pasaron`

Note que la signatura es   `def __init__(self, *args, pension, **kwargs): `aquí` args `captura los parámetros posicionales no nombrados (en este ejemplo `nombre, dia, mes, año`) se pasa también el posicional-nombrado `pension=p `que es capturado en el parámetro `pension` del` __init__ `de `Jubilado`, y los restantes parámentros nombrados no capturados explicitamente (en este ejemplo `salario=s`) se capturan como diccionario con `**kwargs`.

Luego se llama al `__init__` de `Trabajador` que capturará como parámetro posicional nombrado a `salario=s` que se le ha pasado ahora con el valor de `**kargs = {'salario': 12000}`

Como tanto `Jubilado` como `Trabajador `heredan de `Persona` cuando se crea la instancia `Jubilado_Trabajador("Juan", 18, 3, 1979,4000, 12000) `el `__init__ `de esta clase esta llama a

`super().__init__(
            nombre,
            dia,
            mes,
            año,
            pension=p,
            salario=s
        )`

Como `Jubilado` a aparece como primer ancestro y hay un `__init__` definido en `Jubilado` se llama a este `__init__` quien captura los parámetros posicionales como una tupla en su parámetro `*args` el valor `4000` lo captura como valor del parámetro posicional nombrado` pension` y el resto de los parámetros los captura en `**kargs = {'salario': 12000}`. Siguiendo el orden de Python este llama también al `super().__init__` de `Trabajador` ahora con los parámetros `*args = ('Juan', 18, 3, 1979) ``salario: 12000 `**kargs = {} que captura a salario para luego seguir con la llamada `super().__init__ ` con los parámetros` *args = ('Juan', 18, 3, 1979) `que hacen las inicializaciones en `Persona`.

Esto sigue el método conocido como qué **MRO** (_Method Resolution Orden_) que define cómo se llaman o buscan los métodos respectivos en las clases ancestros de una clase en la jerarquía en Python. Esto se ilustrará con más ejemplos en la sección 5.2


In [22]:
#Cómo se podría hacer usando super
from datetime import datetime, date

class Persona:
    def __init__(self, nombre: str, dia: int, mes: int, año: int):
        self.nombre = nombre
        self.fecha_nacimiento = date(año, mes, dia)
        print("__init__ de Persona")
    def edad(self) -> int:
        hoy = datetime.now().date()
        edad = hoy.year - self.fecha_nacimiento.year
        if (hoy.month, hoy.day) < (self.fecha_nacimiento.month, self.fecha_nacimiento.day):
            #estamos en una fecha anterior a la de mi cumpleaños
            edad -= 1
        return edad
    def ingresos(self) -> float:
        return 0.0
    def __str__(self):
        return f"Soy {type(self).__name__} mi nombre es {self.nombre} tengo {self.edad()} años. Mis ingresos son {self.ingresos()}"

class Jubilado(Persona):
    def __init__(self, *args, pension, **kwargs):
        print("__init__ de Jubilado")
        print(f'*args = {args} pension: {pension} **kargs = {kwargs}')
        self.pension = pension
        super().__init__(*args, **kwargs) #paso como diccionario los mismos parámetros nombrados que me pasaron

    def ingresos(self):
        return self.pension

class Trabajador(Persona):
    def __init__(self, *args, salario, **kwargs):
        print("__init__ de Trabajador")
        print(f'*args = {args} salario: {salario} **kargs = {kwargs}')
        self.salario = salario #salario se captura como parámetro posicional
        super().__init__(*args, **kwargs) #paso como diccionario el resto de los parámetros nombrados que me pasaron

    def ingresos(self):
        return self.salario

class Estudiante(Persona):
    def __init__(self, *args, estipendio, **kargs):
        print("__init__ de Estudiante")
        print(f'estipendio: {estipendio} **kargs = {kargs}')
        super().__init__(*args, **kargs) #paso como diccionario el resto de los parámetros nombrados que me pasaron
        self.estipendio = estipendio

    def ingresos(self):
        return self.estipendio

class Jubilado_Trabajador(Jubilado, Trabajador):
    def __init__(self, nombre, dia, mes, año, p, s):
        super().__init__(
            nombre,
            dia,
            mes,
            año,
            pension=p,
            salario=s
        )
        print("__init__ de Jubilado_Trabajador")
    def ingresos(self):
        return self.pension + self.salario

profesor = Jubilado_Trabajador("Juan", 18, 3, 1979,4000, 12000)
print(profesor)
print()

class Estudiante_Trabajador(Estudiante, Trabajador):
    def __init__(self, nombre, dia, mes, año, e, s):
        super().__init__(
            nombre,
            dia,
            mes,
            año,
            estipendio=e,
            salario=s
        )
        print("init de Estudiante_Trabajador")

    def ingresos(self):
        return self.estipendio + self.salario

alumno_ayudante = Estudiante_Trabajador("Ana", 18, 3, 2005, 200, 1000)
print(alumno_ayudante)

__init__ de Jubilado
*args = ('Juan', 18, 3, 1979) pension: 4000 **kargs = {'salario': 12000}
__init__ de Trabajador
*args = ('Juan', 18, 3, 1979) salario: 12000 **kargs = {}
__init__ de Persona
__init__ de Jubilado_Trabajador
Soy Jubilado_Trabajador mi nombre es Juan tengo 47 años. Mis ingresos son 16000

__init__ de Estudiante
estipendio: 200 **kargs = {'salario': 1000}
__init__ de Trabajador
*args = ('Ana', 18, 3, 2005) salario: 1000 **kargs = {}
__init__ de Persona
init de Estudiante_Trabajador
Soy Estudiante_Trabajador mi nombre es Ana tengo 21 años. Mis ingresos son 1200


## 5.2 Orden de Resolución de Métodos (MRO)

En este sección vamos a ilustrar cómo Python decide a cuál metodo invocar cuando de llama a un método ` m ` a través de una instancia de una clase `A2 `que es resultado de una jerarquía de herencia. Comente o descomente las definiciones de `m `del siguiente código para que observe el resultado. Cuando se llama al método m a través de una instancia de` A1_1` Python buscará si existe el tal método` m` en `A1_1`, si no hay una definición buscará en la clase de la cual hereda (en este caso `A1`) y asú sucesivamente en la rama jerárquica hasta encontrar un ancestro en que esté definido m

Pruebe comentando las diferentes definiciones de `m` para que observe el resultado.

In [7]:
class A:
    def m(self):
        print("Soy m en A")
    pass

class A1(A):
    def m(self):
        print("Soy m en A1")
    pass

class A1_1(A1):
    def __init__(self):
        print("He sido creado como instancia de A1_1")
    def m(self):
        print("Soy m en A1_1")
    pass

b = A1_1()
b.m()

# Pruebe ejecutar el código poniendo en comentario las respectivas definiciones de m

He sido creado como instancia de A1_1
Soy m en A1_1


Cómo funciona esto en caso de Herencia Múltiple. En el código a continuación la clase A12_1 hereda múltiple de las clases A1 y A2 si no hay un método m definido en A12_1 Python buscará en el primer padre de A12_1, en este caso A1, si no hay tal método en A1 entonces buscará en las siguientes clases padres, en este caso A2. Si tampoco hay en A2 entonces buscará en A.

Pruebe el código siguiente comentando las definiciones de m para forzar la búsqueda del método m que corresponda. De igual modo pruebe descomentando las correspondientes definiciones de __init__, cuando se ejecuta un super Python lo busca siguiendo la secuencia de niveles en la jerarquía de clases. Comente el __init__ de la clase A2. Note que en este caso la ejecución del código escribiría
`
Inicializando instancia de A123_1
Inicializando instancia de A1
Inicializando instancia de A3
Inicializando instancia de A
Soy m en A123_1`

o sea se ejecuta la secuencia de `super().__init()` saltándose la de `A2` porque no tendría definido `__init__` pero ejecutando el de `A3` antes de ejecutar al `__init__` del padre `A`

In [21]:
class A:
    def __init__(self):
        print("Inicializando instancia de A")
        super().__init__()
    def m(self):
        print("Soy m en A")
    pass

class A1(A):
    def __init__(self):
        print("Inicializando instancia de A1")
        super().__init__()
    def m(self):
        print("Soy m en A1")
    pass

class A2(A):
    def __init__(self):
        print("Inicializando instancia de A2")
        super().__init__()
    def m(self):
        print("Soy m en A2")
    pass

class A3(A):
    def __init__(self):
        print("Inicializando instancia de A3")
        super().__init__()
    def m(self):
        print("Soy m en A3")
    pass

class A123_1(A1, A2, A3):
    def __init__(self):
        print("Inicializando instancia de A123_1")
        super().__init__()
    def m(self):
        print("Soy m en A123_1")
    pass

b = A123_1()
b.m()


Inicializando instancia de A123_1
Inicializando instancia de A1
Inicializando instancia de A3
Inicializando instancia de A
Soy m en A123_1


En la práctica no va a ser frecuente que Ud necesite escribir jerarquías complicadas con herencia múltiple. Los ejemplos que les hemos mostrado aquí son para darle idea de cómo usted puede probar y experimentar por su cuenta

## 5.3  EJERCICIOS
1. Defina una clase `Accion` con un método abstracto `queHago` que devuelva un string y que añada este string cuando se aplique el `__str__ `a una instancia de una clase que también herede de `Accion`. Así por ejemplo si las clases` Gato` y `Leon` definidas en la conferencia 4 heredan de `Animal `pero también de `Accion ` entonces al ejecutar

`class Gato(Animal, Accion):`

`...`

`class Leon(Animal, Accion):`

`...`

`g = Gato("Tom") `

`print(g) `

`l = Leon("Simba") `

`print(l)`

se escriba

`Soy un Gato de nombre Tom que hace Miau, Miau y cazo ratones
Soy un Leon de nombre Simba que hace Gouarr, Gouarr y cazo gacelas`

2. Ud puede preguntarse por qué no poner toda la lógica de la clase `Accion` dentro de la clase `Animal` base de la jerarquía para que todos los animales la herden y evitamos tener que hacer herencia múltiple. Suponga que queremos lograr algo similar en la jerarquía de `Persona` y que el código de

`class AlumnoAyudante(Estudiante, Trabajador, Accion):`

`...
`
`aa = AlumnoAyudante("Ana", 18, 3, 2005, 200, 1000, "Doy clases de Python")`

`print(aa)`

escriba

`Soy AlumnoAyudante mi nombre es Ana tengo 21 años. Mis ingresos son 1200. Doy clases de Python`

Es decir heredando también múltiple de `Accion`, pero sin tener que meter la lógica de `Accion` dentro de la clase `Persona` sino que `AlumnoAyudante `solo tenga que implementar el método `queHago`, lo cual es su obligación por heredar de `Accion`

(C) MKM para MATCOM UH

In [8]:
#Cómo se podría hacer usando super
from datetime import datetime, date

class Persona:
    def __init__(self, nombre: str, dia: int, mes: int, año: int):
        self.nombre = nombre
        self.fecha_nacimiento = date(año, mes, dia)
        print("__init__ de Persona")
    def edad(self) -> int:
        hoy = datetime.now().date()
        edad = hoy.year - self.fecha_nacimiento.year
        if (hoy.month, hoy.day) < (self.fecha_nacimiento.month, self.fecha_nacimiento.day):
            #estamos en una fecha anterior a la de mi cumpleaños
            edad -= 1
        return edad
    def ingresos(self) -> float:
        return 0.0
    def __str__(self):
        return f"Soy {type(self).__name__} mi nombre es {self.nombre} tengo {self.edad()} años. Mis ingresos son {self.ingresos()}"

class Jubilado(Persona):
    def __init__(self, *args, pension, **kwargs):
        print("__init__ de Jubilado")
        print(f'*args = {args} pension: {pension} **kargs = {kwargs}')
        self.pension = pension
        super().__init__(*args, **kwargs) #paso como diccionario los mismos parámetros nombrados que me pasaron

    def ingresos(self):
        return self.pension

class Trabajador(Persona):
    def __init__(self, *args, salario, **kwargs):
        print("__init__ de Trabajador")
        print(f'*args = {args} salario: {salario} **kargs = {kwargs}')
        self.salario = salario #salario se captura como parámetro posicional
        super().__init__(*args, **kwargs) #paso como diccionario el resto de los parámetros nombrados que me pasaron

    def ingresos(self):
        return self.salario

class Jubilado_Trabajador(Jubilado, Trabajador):
    def __init__(self, nombre, dia, mes, año, p, s):
        super().__init__(
            nombre,
            dia,
            mes,
            año,
            pension=p,
            salario=s
        )
        print("__init__ de Jubilado_Trabajador")
    def ingresos(self):
        return self.pension + self.salario

profesor = Jubilado_Trabajador("Juan", 18, 3, 1979,4000, 12000)
print(profesor)
print()

__init__ de Jubilado
*args = ('Juan', 18, 3, 1979) pension: 4000 **kargs = {'salario': 12000}
__init__ de Trabajador
*args = ('Juan', 18, 3, 1979) salario: 12000 **kargs = {}
__init__ de Persona
__init__ de Jubilado_Trabajador
Soy Jubilado_Trabajador mi nombre es Juan tengo 47 años. Mis ingresos son 16000



In [57]:
class Actividad:
    def __init__(self, accion:str):
        self.accion = accion
    def QueHago(self):
        return f"{self.accion}"

class AlumnoAyudante(Estudiante_Trabajador, Actividad):
    def __init__(self, nombre, dia, mes, año, e, s, a):
        super().__init__(
            nombre,
            dia,
            mes,
            año,
            e,
            s
        )
        self.accion = a

    def __str__(self):
        return f"{super().__str__()} {self.QueHago()}"

aa = AlumnoAyudante("Ana", 18, 3, 2005, 200, 1000, a =" Doy clases de programacion")
print(aa)



__init__ de Estudiante
estipendio: 200 **kargs = {'salario': 1000}
__init__ de Trabajador
salario: 1000 **kargs = {}
__init__ de Persona
init de Estudiante_Trabajador
Soy AlumnoAyudante mi nombre es Ana tengo 21 años. Mis ingresos son 1200 +  Doy clases de programacion


In [ ]:
def Pruebakargs(**kargs):
    print(kargs)
Pruebakargs(100, 2, 3, 4)

